# 02 — Depletion projection (Colab)

**Task:** Predict next-year saturated thickness per county, then iterate forward for the 25–100-year projection. Short-horizon target is one-step-ahead `y_next_thickness`. Same three GBDT frameworks + stacking ensemble pattern as `01_tx_extraction_imputation.ipynb`.

---

### Honest take on the modeling choice

This is a **time-series forecasting** problem with short per-county histories (~30 pts each). A few notes before grinding on hyperparams:

1. **Per-county linear regression is a shockingly strong baseline.** Aquifer decline is locally near-linear over a 5–15 year window. If a pooled GBDT can't clearly beat per-county OLS, the extra complexity isn't buying us anything — keep the linear.
2. **The long-horizon projection (25–100y) is not an ML problem.** Beyond ~15 years, short-window extrapolation dominates any model's bias. The right tool there is a Deines-style mass balance — see `aquiferwatch/analytics/depletion.py`. Treat this notebook as the short-horizon (≤15y) workhorse, not the century forecast.
3. **Lag features leak across counties if you split rows randomly.** We use GroupKFold on FIPS, same as notebook 01.
4. **Recursive multi-step prediction accumulates error fast.** For the 25-year view, prefer a direct multi-output model (one regressor per horizon) or physics-informed extrapolation. Recursive GBDT for 25y is a known trap — the 2024 Makridakis M6 showed it lagging even naïve baselines at h=25.

### Anti-patterns

- Don't fit a GBDT on 30 points per county (per-county GBDT). Use a pooled model.
- Don't include current-year thickness as a feature — that's leakage. Only use lags ≥1.
- Don't average 2100 projections across this model and a mass-balance model — different error regimes, the average is worse than either.


## 0. Colab bootstrap

In [ ]:
# Install aquiferwatch WITHOUT its transitive deps so we don't downgrade
# Colab's pre-installed IPython / jupyter (known to break %load_ext autoreload).
# Colab already provides numpy, pandas, pyarrow, scipy, sklearn, matplotlib.
!pip install -q --no-deps git+https://github.com/Vigneshwarr3/FusionHack_2026.git
!pip install -q lightgbm xgboost catboost mlflow boto3 pydantic-settings


In [ ]:
from aquiferwatch.colab import bootstrap
bootstrap(team_member="raj")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

from aquiferwatch.analytics.models import (
    BoostedRegressor,
    StackingEnsemble,
    train_and_log,
    spatial_cv_scores,
    summarize_cv,
    evaluate,
)

# Toggle the data source here. `True` pulls the real per-county annual
# thickness panel from S3-backed parquets (TWDB TX + KGS WIZARD KS).
# `False` uses the synthetic generator for pipeline smoke tests.
USE_REAL_DATA = True

if USE_REAL_DATA:
    from aquiferwatch.data.real import (
        build_depletion_history,
        build_lagged_features,
        REAL_DEPLETION_FEATURE_COLS as FEATURE_COLS,
        REAL_DEPLETION_TARGET_COL as TARGET_COL,
    )
else:
    from aquiferwatch.data.synthetic_depletion import (
        generate_thickness_history,
        build_lagged_features,
        FEATURE_COLS,
        TARGET_COL,
    )


## 1. Load data

Set `USE_REAL_DATA = True` in the imports cell above (default) to pull the
real per-county annual saturated-thickness panel built from TWDB TX wells
(pre-computed `sat_thickness_m`) + KGS WIZARD KS wells (thickness = bedrock
depth − depth-to-water, joined against `kgs_county_bedrock.parquet`).

The loader pulls from `data/processed/` locally, or transparently fetches
missing parquets from the shared S3 bucket pinned in repo-root `DATA_VERSION`.
Set the toggle to `False` for synthetic-data smoke tests.


In [ ]:
if USE_REAL_DATA:
    history = build_depletion_history(min_year=1990, min_points_per_county=6)
    print(f'real panel: {history["fips"].nunique()} counties × {history["year"].nunique()} years')
else:
    history = generate_thickness_history(n_counties=120, n_years=35, seed=17)
    print(f'synthetic panel: {history["fips"].nunique()} counties × {history["year"].nunique()} years')

feat = build_lagged_features(history)
print(f'history rows={len(history)}  modeling rows={len(feat)}')
feat.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for f in history["fips"].unique()[:20]:
    sub = history[history["fips"] == f]
    ax.plot(sub["year"], sub["saturated_thickness_m"], alpha=0.4)
ax.set_title("sample saturated-thickness trajectories")
ax.set_xlabel("year")
ax.set_ylabel("thickness (m)")
plt.show()

## 2. Spatial split — whole-county hold-out

In [ ]:
rng = np.random.default_rng(0)
counties = feat["fips"].unique()
rng.shuffle(counties)
test_counties = set(counties[: int(0.2 * len(counties))])

train_df = feat[~feat["fips"].isin(test_counties)].copy()
test_df  = feat[ feat["fips"].isin(test_counties)].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET_COL]
groups_train = train_df["fips"]
print(f"train rows={len(X_train)}  test rows={len(X_test)}")

## 3. Baseline — per-county OLS

The model every GBDT must beat. If it doesn't, drop the GBDT and ship the linear.

In [ ]:
# Fit one linear regression per county on (year, lag1) -> next thickness.
baseline_preds = []
for fips, sub in train_df.groupby("fips"):
    if len(sub) < 4:
        continue
    lr = LinearRegression().fit(sub[["thickness_lag1"]], sub[TARGET_COL])
    test_sub = test_df[test_df["fips"] == fips]
    if len(test_sub):
        baseline_preds.append(pd.DataFrame({
            "fips": fips,
            "y_true": test_sub[TARGET_COL].values,
            "y_pred": lr.predict(test_sub[["thickness_lag1"]]).ravel(),
        }))

# For held-out counties we have no per-county model. Use the global mean slope as fallback.
# (In real use this is where the pooled model shines.)
global_lr = LinearRegression().fit(train_df[["thickness_lag1"]], train_df[TARGET_COL])
unseen = test_df[~test_df["fips"].isin({p["fips"].iloc[0] for p in baseline_preds if len(p)})]
if len(unseen):
    baseline_preds.append(pd.DataFrame({
        "fips": unseen["fips"].values,
        "y_true": unseen[TARGET_COL].values,
        "y_pred": global_lr.predict(unseen[["thickness_lag1"]]).ravel(),
    }))

preds = pd.concat(baseline_preds, ignore_index=True)
baseline_metrics = evaluate(preds["y_true"], preds["y_pred"].values)
print("per-county OLS baseline:", baseline_metrics)

## 4. LightGBM

In [ ]:
lgb_params = {
    "n_estimators": 500,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "verbose": -1,
}
lgb_model = BoostedRegressor("lightgbm", params=lgb_params)
_ = train_and_log(lgb_model, X_train, y_train, X_test, y_test,
                  module="depletion", run_name="dep_lightgbm_baseline")

## 5. XGBoost

In [ ]:
xgb_params = {
    "n_estimators": 500,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
}
xgb_model = BoostedRegressor("xgboost", params=xgb_params)
_ = train_and_log(xgb_model, X_train, y_train, X_test, y_test,
                  module="depletion", run_name="dep_xgboost_baseline")

## 6. CatBoost

In [ ]:
cat_params = {
    "iterations": 700,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "verbose": 0,
}
cat_model = BoostedRegressor("catboost", params=cat_params)
_ = train_and_log(cat_model, X_train, y_train, X_test, y_test,
                  module="depletion", run_name="dep_catboost_baseline")

## 7. Stacking ensemble (group CV)

In [ ]:
stack = StackingEnsemble(
    base=[
        BoostedRegressor("lightgbm", params=lgb_params),
        BoostedRegressor("xgboost",  params=xgb_params),
        BoostedRegressor("catboost", params=cat_params),
    ],
    meta="ridge",
    n_folds=5,
)
_ = train_and_log(stack, X_train, y_train, X_test, y_test,
                  module="depletion", run_name="dep_stack_lgb_xgb_cat_ridge",
                  groups_train=groups_train)

## 8. Spatial CV comparison

In [ ]:
factories = {
    "lightgbm": lambda: BoostedRegressor("lightgbm", params=lgb_params),
    "xgboost":  lambda: BoostedRegressor("xgboost",  params=xgb_params),
    "catboost": lambda: BoostedRegressor("catboost", params=cat_params),
}
rows = []
for name, factory in factories.items():
    cv = spatial_cv_scores(factory, X_train, y_train, groups=groups_train, n_folds=5)
    s = summarize_cv(cv)
    s.name = name
    rows.append(s)
summary = pd.concat(rows, axis=1).T
summary[["mae_mean", "mae_std", "rmse_mean", "r2_mean"]].round(3)

## 9. Decision log

- **Per-county OLS beaten?:** _yes/no and by how much on MAE_
- **Best pooled model:** _…_
- **Ready to hand to the scenario engine?:** _…_
